# Homework 3 – Portfolio Optimization via Modern Portfolio Theory

**Framework:** Markowitz Mean-Variance · Black-Litterman · Fama-French 3-Factor  
**Asset Universe:** Crypto · Equities · ETF/VC Proxy  
**Data:** yfinance (Bloomberg API structure provided as alternative)  
**Predictive horizon:** 254 ore (≈ 10.6 giorni 24/7 · ≈ 39 trading days)  
**Rebalancing:** ogni 254 ore tramite pesi posteriori Black-Litterman

---

### Struttura del notebook
| Sezione | Contenuto |
|---------|-----------|
| **1. Data Collection** | Fetch orario, log-return, statistiche descrittive |
| **2. EDA** | Prezzi, distribuzioni, correlazioni, volatilità rolling |
| **3. Predictive Model** | ARIMA(1,1,1) rendimenti attesi · GARCH(1,1) volatilità |
| **4. Markowitz** | Frontiera Efficiente · Global Minimum Variance Portfolio |
| **5. Black-Litterman** | Prior CAPM + Views ARIMA/GARCH → pesi posteriori |
| **6. Fama-French** | Regressione 3-fattori per separare α da rischio sistematico |
| **7. Monte Carlo** | 10 000 portafogli simulati · stress test |
| **8. Rebalancing Engine** | Ciclo 254h con Sortino Ratio e performance dashboard |

## ⚠️  Nota sull'API Bloomberg

Il modulo `blpapi` (Bloomberg API) richiede una **licenza Bloomberg Terminal** attiva.  
Il codice Bloomberg è fornito in blocchi commentati per ogni sezione.  
**Per eseguire questo notebook installa le dipendenze:**

```bash
pip install yfinance pandas-datareader arch scipy statsmodels matplotlib seaborn
```

In [4]:
# ── Imports ───────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import io, zipfile, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from datetime import datetime, timedelta
from itertools import product

# Finance data
import yfinance as yf

# Statistical / time-series models
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from arch import arch_model

# Optimization
from scipy.optimize import minimize
from scipy.stats import norm

# ── Fama-French helper (replaces pandas_datareader broken on Py 3.14) ──────────
def fetch_ff3_daily(start='2020-01-01'):
    """Download FF3 daily factors directly from Ken French's website."""
    url = ('https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/'
           'F-F_Research_Data_Factors_daily_CSV.zip')
    try:
        r = requests.get(url, timeout=20)
        r.raise_for_status()
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            fname = [n for n in z.namelist() if n.endswith('.CSV')][0]
            with z.open(fname) as f:
                raw = f.read().decode('utf-8', errors='replace')
        # Skip header rows until data starts
        lines = raw.splitlines()
        data_start = next(i for i, l in enumerate(lines) if l.strip() and l.strip()[0].isdigit())
        data_end   = next((i for i, l in enumerate(lines[data_start:], data_start)
                           if l.strip() == ''), len(lines))
        df = pd.read_csv(io.StringIO('\n'.join(lines[data_start:data_end])),
                         header=None, names=['Date','Mkt-RF','SMB','HML','RF'])
        df['Date'] = pd.to_datetime(df['Date'].astype(str), format='%Y%m%d')
        df = df.set_index('Date') / 100
        return df[df.index >= start]
    except Exception as e:
        print(f"FF3 download failed: {e}")
        return None

plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 12})
sns.set_palette('husl')
pd.set_option('display.float_format', '{:.5f}'.format)

print("All imports OK ✓")

All imports OK ✓


In [5]:
# ── Asset Universe & Global Parameters ────────────────────────────────────────

ASSETS = {
    'Crypto':  ['BTC-USD', 'ETH-USD', 'SOL-USD'],
    'Stocks':  ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN'],
    'ETF_VC':  ['SPY', 'QQQ', 'ARKK'],   # ARKK = VC/innovation proxy
}

ALL_TICKERS = [t for group in ASSETS.values() for t in group]
N_ASSETS    = len(ALL_TICKERS)

# ── Market-cap weights (approximate, used as Black-Litterman prior) ────────────
# BTC ETH SOL  AAPL  MSFT  NVDA  GOOGL AMZN  SPY   QQQ   ARKK
MKT_CAPS = np.array([
    1.20e12, 4.00e11, 8.00e10,   # Crypto
    3.20e12, 3.10e12, 2.80e12, 2.10e12, 1.80e12,  # Stocks
    5.00e11, 3.00e11, 8.00e9,   # ETF/VC
])
W_MKT = MKT_CAPS / MKT_CAPS.sum()   # normalised prior weights

# ── Time & risk parameters ─────────────────────────────────────────────────────
REBALANCE_HOURS  = 254        # rebalancing window
RISK_FREE_ANNUAL = 0.0525     # Fed Funds ~5.25 %
HOURS_PER_YEAR   = 8_760      # crypto trades 24/7
RF_HOURLY        = (1 + RISK_FREE_ANNUAL) ** (1 / HOURS_PER_YEAR) - 1

TAU          = 0.05           # Black-Litterman: prior uncertainty scalar
DELTA        = 2.5            # implied risk-aversion coefficient
N_MC         = 10_000         # Monte Carlo portfolios

print(f"Universe  : {N_ASSETS} assets — {list(ASSETS.keys())}")
print(f"Rebalance : every {REBALANCE_HOURS} hours")
print(f"RF hourly : {RF_HOURLY:.8f}  ({RISK_FREE_ANNUAL*100:.2f} % annual)")

Universe  : 11 assets — ['Crypto', 'Stocks', 'ETF_VC']
Rebalance : every 254 hours
RF hourly : 0.00000584  (5.25 % annual)


---
## 1. Data Collection

Fonte primaria: **yfinance** (gratuito, nessuna licenza).  
La struttura Bloomberg equivalente è mostrata in commento.

> **Bloomberg equivalent** (richiede Terminal + `blpapi`):
> ```python
> import blpapi
> session = blpapi.Session()
> session.start()
> refDataService = session.getService("//blp/refdata")
> request = refDataService.createRequest("IntradayBarRequest")
> request.set("security", "AAPL US Equity")
> request.set("eventType", "TRADE")
> request.set("interval", 60)             # 60-min bars
> request.set("startDateTime", blpapi.Datetime(2023, 1, 1, 0, 0, 0))
> request.set("endDateTime",   blpapi.Datetime(2024, 12, 31, 23, 59, 59))
> session.sendRequest(request)
> ```

In [6]:
# ── 1.1  Fetch hourly OHLCV (max 730 days via yfinance) ───────────────────────
print("Downloading hourly price data …")
raw_hourly = yf.download(
    tickers     = ALL_TICKERS,
    period      = '730d',
    interval    = '1h',
    auto_adjust = True,
    progress    = False,
)
prices_h = raw_hourly['Close'].copy()
# Remove timezone (UTC on hourly data)
if prices_h.index.tz is not None:
    prices_h.index = prices_h.index.tz_convert(None)

# Drop columns that are all-NaN (asset not available for full period)
prices_h = prices_h.dropna(axis=1, how='all')
prices_h = prices_h.ffill().bfill()

# Keep only tickers successfully downloaded
ALL_TICKERS = list(prices_h.columns)
N_ASSETS    = len(ALL_TICKERS)

# ── 1.2  Fetch daily prices (5 years) for Fama-French regression ──────────────
print("Downloading daily price data …")
raw_daily = yf.download(
    tickers     = ALL_TICKERS,
    period      = '5y',
    interval    = '1d',
    auto_adjust = True,
    progress    = False,
)
prices_d = raw_daily['Close'].copy()
if prices_d.index.tz is not None:
    prices_d.index = prices_d.index.tz_convert(None)
prices_d = prices_d[ALL_TICKERS].ffill().bfill()

# ── 1.3  Log returns ──────────────────────────────────────────────────────────
log_ret_h = np.log(prices_h / prices_h.shift(1)).dropna()
log_ret_d = np.log(prices_d / prices_d.shift(1)).dropna()

# Update market-cap weights to match available tickers
MKT_CAPS_map = dict(zip(
    ['BTC-USD','ETH-USD','SOL-USD','AAPL','MSFT','NVDA','GOOGL','AMZN','SPY','QQQ','ARKK'],
    [1.20e12, 4.00e11, 8.00e10, 3.20e12, 3.10e12, 2.80e12, 2.10e12, 1.80e12, 5.00e11, 3.00e11, 8.00e9]
))
caps = np.array([MKT_CAPS_map.get(t, 1e10) for t in ALL_TICKERS])
W_MKT    = caps / caps.sum()
BOUNDS   = tuple((0.0, 0.40) for _ in range(N_ASSETS))

print(f"\nAssets available : {N_ASSETS} — {ALL_TICKERS}")
print(f"Hourly prices : {prices_h.shape}   →  log-returns: {log_ret_h.shape}")
print(f"Daily  prices : {prices_d.shape}   →  log-returns: {log_ret_d.shape}")
print(f"Date range (hourly): {log_ret_h.index[0]}  →  {log_ret_h.index[-1]}")
print(f"Date range (daily) : {log_ret_d.index[0].date()}  →  {log_ret_d.index[-1].date()}")


6 Failed downloads:
['ETH-USD', 'SOL-USD', 'BTC-USD', 'MSFT', 'NVDA', 'GOOGL']: TypeError("'NoneType' object is not subscriptable")



Assets available : 5 — ['AAPL', 'AMZN', 'ARKK', 'QQQ', 'SPY']
Hourly prices : (5078, 5)   →  log-returns: (5077, 5)
Daily  prices : (1256, 5)   →  log-returns: (1255, 5)
Date range (hourly): 2023-05-01 14:30:00  →  2026-03-27 19:30:00
Date range (daily) : 2021-03-30  →  2026-03-27


In [7]:
# ── 1.4  Descriptive statistics (hourly log-returns) ─────────────────────────
stats = log_ret_h.describe().T
stats['skewness'] = log_ret_h.skew()
stats['kurtosis'] = log_ret_h.kurtosis()
stats['ann_vol']  = log_ret_h.std() * np.sqrt(HOURS_PER_YEAR)  # annualised volatility
stats['ann_ret']  = log_ret_h.mean() * HOURS_PER_YEAR          # annualised mean return

display(stats[['mean', 'std', 'skewness', 'kurtosis', 'ann_ret', 'ann_vol']].round(5))

,mean,std,skewness,kurtosis,ann_ret,ann_vol
Ticker,,,,,,
AAPL,0.00008,0.00606,-0.36609,21.61285,0.66560,0.56691
AMZN,0.00013,0.00775,-0.19282,42.90411,1.12575,0.72533
ARKK,0.00012,0.00926,-0.05235,11.73598,1.02494,0.86701
QQQ,0.00011,0.00456,-0.00071,15.42279,0.96358,0.42693
SPY,0.00008,0.00354,-0.08981,24.80622,0.72729,0.33164


---
## 2. Exploratory Data Analysis

In [ ]:
# ── 2.1  Normalised price series (base 100) ───────────────────────────────────
# Keep only asset classes that have at least one available ticker
ASSETS_avail = {cls: [t for t in tks if t in prices_h.columns]
                for cls, tks in ASSETS.items()}
ASSETS_avail = {k: v for k, v in ASSETS_avail.items() if v}

colors = plt.cm.tab10.colors
n_cls  = len(ASSETS_avail)

fig, axes = plt.subplots(n_cls, 1, figsize=(14, 4 * n_cls), squeeze=False)
for ax, (cls, tickers) in zip(axes[:, 0], ASSETS_avail.items()):
    for i, t in enumerate(tickers):
        s      = prices_h[t].dropna()
        s_norm = s / s.iloc[0] * 100
        ax.plot(s_norm.index, s_norm.values, label=t, linewidth=1.2,
                color=colors[i % len(colors)])
    ax.set_title(f'{cls} — Normalised prices (base 100)')
    ax.set_ylabel('Index')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle('Asset Universe — Normalised Price Series (Hourly)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.2  Return distributions + correlation heatmap ──────────────────────────
fig = plt.figure(figsize=(16, 6))
gs  = gridspec.GridSpec(1, 2, width_ratios=[1.4, 1])

# Distributions
ax1 = fig.add_subplot(gs[0])
for t in ALL_TICKERS:
    r = log_ret_h[t].dropna()
    r.plot.kde(ax=ax1, label=t, linewidth=1.1)
ax1.set_xlim(-0.05, 0.05)
ax1.set_title('Hourly Log-Return Distributions (KDE)')
ax1.set_xlabel('Log return')
ax1.legend(fontsize=7, ncol=2)
ax1.grid(alpha=0.3)

# Correlation heatmap
ax2 = fig.add_subplot(gs[1])
corr = log_ret_h.corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, ax=ax2, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.4, annot_kws={'size': 7},
            cbar_kws={'shrink': 0.8})
ax2.set_title('Correlation Matrix (Hourly Returns)')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# ── 2.3  Rolling 7-day annualised volatility ──────────────────────────────────
ROLL_H   = 7 * 24
roll_vol = log_ret_h.rolling(ROLL_H).std() * np.sqrt(HOURS_PER_YEAR)

n_cls  = len(ASSETS_avail)
fig, axes = plt.subplots(n_cls, 1, figsize=(14, 4 * n_cls), squeeze=False)
for ax, (cls, tickers) in zip(axes[:, 0], ASSETS_avail.items()):
    for i, t in enumerate(tickers):
        ax.plot(roll_vol[t].dropna(), label=t, linewidth=1.1,
                color=colors[i % len(colors)])
    ax.set_title(f'{cls} — 7-Day Rolling Annualised Volatility')
    ax.set_ylabel('σ (annualised)')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle('Rolling Volatility by Asset Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Predictive Model — ARIMA(1,1,1) + GARCH(1,1) (254-Hour Horizon)

**Razionale:**  
- **ARIMA(1,1,1)** cattura autocorrelazione nei log-prezzi e produce la stima del rendimento atteso μᵢ su 254 ore  
- **GARCH(1,1)** modella la volatilità condizionata σᵢ (clustering della volatilità), usata come incertezza delle Views in Black-Litterman (matrice Ω)  
- I rendimenti finanziari sono vicini al *random walk*: l'ARIMA produce previsioni di media "deboli" ma la stima della varianza condizionata tramite GARCH è robusta e ampiamente validata in letteratura

In [ ]:
# ── 3.1  ARIMA(1,1,1) — 254h cumulative return forecast per asset ──────────────
# Fit on last 500 hourly observations (fast) — scale up to full series if needed
TRAIN_OBS   = 500
HORIZON     = REBALANCE_HOURS   # 254 steps ahead

arima_mu    = {}   # mean return forecast over 254h
arima_sigma = {}   # forecast std dev over 254h

for ticker in ALL_TICKERS:
    series = log_ret_h[ticker].dropna().values[-TRAIN_OBS:]
    try:
        model  = ARIMA(series, order=(1, 1, 1))
        fitted = model.fit(method_kwargs={'warn_convergence': False})
        fc     = fitted.get_forecast(steps=HORIZON)
        mu_254 = fc.predicted_mean.sum()          # cumulative expected return
        se_254 = np.sqrt((fc.se_mean ** 2).sum()) # combined std error
    except Exception:
        # Fallback: sample mean / std
        mu_254 = series.mean() * HORIZON
        se_254 = series.std()  * np.sqrt(HORIZON)

    arima_mu[ticker]    = mu_254
    arima_sigma[ticker] = se_254
    print(f"  {ticker:12s}  μ_254h = {mu_254:+.5f}   σ_254h = {se_254:.5f}")

mu_vec = np.array([arima_mu[t] for t in ALL_TICKERS])   # shape (N,)

In [ ]:
# ── 3.2  GARCH(1,1) — conditional volatility forecast (254h) ─────────────────
garch_var = {}   # conditional variance forecast over 254h

for ticker in ALL_TICKERS:
    series = log_ret_h[ticker].dropna().values[-TRAIN_OBS:] * 100  # % for arch
    try:
        gm    = arch_model(series, vol='Garch', p=1, q=1, dist='Normal', rescale=False)
        res   = gm.fit(disp='off', show_warning=False)
        fc    = res.forecast(horizon=HORIZON, reindex=False)
        # variance = mean of all h-step ahead forecasts (in %)
        var_254 = fc.variance.values[-1].mean() / 1e4   # back to decimal
    except Exception:
        var_254 = (log_ret_h[ticker].std() ** 2) * HORIZON

    garch_var[ticker] = var_254

# Diagonal Omega matrix for Black-Litterman (view uncertainty)
omega_diag = np.array([garch_var[t] for t in ALL_TICKERS])
print("GARCH conditional variance (254h forecast):")
for t, v in garch_var.items():
    print(f"  {t:12s}  var = {v:.6f}  |  σ = {np.sqrt(v):.5f}")

---
## 4. Markowitz Mean-Variance Optimization

**Frontiera Efficiente** = insieme dei portafogli che massimizzano il rendimento atteso a parità di varianza (o minimizzano la varianza a parità di rendimento).  
**Global Minimum Variance Portfolio (GMVP)** = punto più a sinistra della frontiera: minimizza matematicamente `w'Σw` senza vincoli sul rendimento atteso.

In [ ]:
# ── 4.1  Covariance matrix (annualised, from hourly returns) ──────────────────
Sigma_h   = log_ret_h.cov().values                  # hourly cov
Sigma_ann = Sigma_h * HOURS_PER_YEAR                # annualised cov
mu_ann    = log_ret_h.mean().values * HOURS_PER_YEAR # annualised mean returns

# ── 4.2  Portfolio utility functions ─────────────────────────────────────────
def port_return(w, mu):   return w @ mu
def port_vol(w, Sigma):   return np.sqrt(w @ Sigma @ w)
def port_sharpe(w, mu, Sigma, rf=RISK_FREE_ANNUAL):
    return (port_return(w, mu) - rf) / port_vol(w, Sigma)

BOUNDS      = tuple((0.0, 0.40) for _ in range(N_ASSETS))  # no short, max 40% per asset
CONSTRAINTS = [{'type': 'eq', 'fun': lambda w: w.sum() - 1}]

# ── 4.3  Global Minimum Variance Portfolio ────────────────────────────────────
def neg_portfolio_var(w, Sigma):
    return w @ Sigma @ w   # minimise variance

res_gmvp = minimize(
    neg_portfolio_var,
    x0     = np.ones(N_ASSETS) / N_ASSETS,
    args   = (Sigma_ann,),
    method = 'SLSQP',
    bounds = BOUNDS,
    constraints = CONSTRAINTS,
    options = {'maxiter': 500}
)
w_gmvp = res_gmvp.x

print("══ Global Minimum Variance Portfolio (GMVP) ══")
print(f"  Annualised Return : {port_return(w_gmvp, mu_ann)*100:.2f} %")
print(f"  Annualised Vol    : {port_vol(w_gmvp, Sigma_ann)*100:.2f} %")
print(f"  Sharpe Ratio      : {port_sharpe(w_gmvp, mu_ann, Sigma_ann):.4f}\n")
pd.Series(w_gmvp, index=ALL_TICKERS).round(4).rename('GMVP weight').to_frame().T

In [ ]:
# ── 4.4  Efficient Frontier (parametric sweep on target return) ───────────────
target_returns = np.linspace(mu_ann.min(), mu_ann.max(), 60)
ef_vols, ef_rets = [], []

for tgt in target_returns:
    cons = CONSTRAINTS + [{'type': 'eq', 'fun': lambda w, t=tgt: port_return(w, mu_ann) - t}]
    res  = minimize(neg_portfolio_var, np.ones(N_ASSETS)/N_ASSETS,
                    args=(Sigma_ann,), method='SLSQP',
                    bounds=BOUNDS, constraints=cons,
                    options={'maxiter': 500})
    if res.success:
        ef_vols.append(port_vol(res.x, Sigma_ann))
        ef_rets.append(tgt)

ef_vols = np.array(ef_vols)
ef_rets = np.array(ef_rets)

# ── Max Sharpe (Tangency) Portfolio ──────────────────────────────────────────
def neg_sharpe(w, mu, Sigma):
    return -port_sharpe(w, mu, Sigma)

res_msr = minimize(neg_sharpe, np.ones(N_ASSETS)/N_ASSETS,
                   args=(mu_ann, Sigma_ann), method='SLSQP',
                   bounds=BOUNDS, constraints=CONSTRAINTS,
                   options={'maxiter': 500})
w_msr = res_msr.x

print(f"Max Sharpe Portfolio  →  Return={port_return(w_msr,mu_ann)*100:.2f}%  "
      f"Vol={port_vol(w_msr,Sigma_ann)*100:.2f}%  "
      f"Sharpe={port_sharpe(w_msr,mu_ann,Sigma_ann):.4f}")

In [ ]:
# ── 4.5  Plot Efficient Frontier ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 7))

ax.plot(ef_vols * 100, ef_rets * 100, 'b-', linewidth=2.5, label='Efficient Frontier')

# Individual assets
for i, t in enumerate(ALL_TICKERS):
    ax.scatter(np.sqrt(Sigma_ann[i, i]) * 100, mu_ann[i] * 100,
               s=60, zorder=5, alpha=0.85)
    ax.annotate(t, (np.sqrt(Sigma_ann[i, i]) * 100, mu_ann[i] * 100),
                fontsize=8, ha='left', va='bottom',
                xytext=(4, 2), textcoords='offset points')

# GMVP
ax.scatter(port_vol(w_gmvp, Sigma_ann)*100, port_return(w_gmvp, mu_ann)*100,
           s=200, marker='*', color='green', zorder=6, label='GMVP')

# Max Sharpe
ax.scatter(port_vol(w_msr, Sigma_ann)*100, port_return(w_msr, mu_ann)*100,
           s=200, marker='D', color='red', zorder=6, label='Max Sharpe (Tangency)')

ax.set_xlabel('Annualised Volatility (%)')
ax.set_ylabel('Annualised Return (%)')
ax.set_title('Markowitz Efficient Frontier', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 5. Black-Litterman Model

**Problema di Markowitz:** i pesi ottimali sono estremamente sensibili alle stime di μ — piccoli errori producono allocazioni estreme.

**Soluzione Black-Litterman:**
1. **Prior (Π):** rendimenti di equilibrio impliciti dal CAPM — `Π = δ · Σ · w_mkt`
2. **Views (P, Q):** le previsioni ARIMA a 254 ore come *views* assolute su ogni asset
3. **Incertezza Views (Ω):** matrice diagonale con le varianze GARCH condizionate
4. **Posterior:** media ponderata bayesiana tra Prior e Views

$$\mu_{BL} = \left[(\tau\Sigma)^{-1} + P^\top\Omega^{-1}P\right]^{-1}\left[(\tau\Sigma)^{-1}\Pi + P^\top\Omega^{-1}Q\right]$$

In [ ]:
# ── 5.1  Black-Litterman implementation ──────────────────────────────────────

def black_litterman(w_mkt, Sigma, mu_views, var_views, tau=TAU, delta=DELTA):
    """
    Black-Litterman posterior expected returns.

    Parameters
    ----------
    w_mkt      : (N,)   market-cap weights (prior)
    Sigma      : (N,N)  annualised covariance matrix
    mu_views   : (N,)   absolute views on each asset (ARIMA forecasts, annualised)
    var_views  : (N,)   variance of each view (GARCH, annualised)
    tau        : scalar uncertainty on prior (typically 0.01–0.10)
    delta      : scalar implied risk-aversion coefficient

    Returns
    -------
    mu_bl      : (N,)   posterior expected returns
    Sigma_bl   : (N,N)  posterior covariance
    """
    N   = len(w_mkt)
    Pi  = delta * Sigma @ w_mkt           # CAPM equilibrium returns (prior)
    P   = np.eye(N)                        # absolute views: one per asset
    Q   = mu_views                         # view returns
    Omega = np.diag(var_views)             # view uncertainty (diagonal)
    tS  = tau * Sigma

    # Posterior precision-weighted average
    tS_inv    = np.linalg.inv(tS)
    Omega_inv = np.linalg.inv(Omega)
    M_inv     = tS_inv + P.T @ Omega_inv @ P
    M         = np.linalg.inv(M_inv)

    mu_bl    = M @ (tS_inv @ Pi + P.T @ Omega_inv @ Q)
    Sigma_bl = Sigma + M

    return mu_bl, Sigma_bl, Pi

# ── Annualise ARIMA views (254h → annual) ────────────────────────────────────
mu_views_ann  = mu_vec * (HOURS_PER_YEAR / HORIZON)  # scale to annual
var_views_ann = omega_diag * (HOURS_PER_YEAR / HORIZON)

mu_bl, Sigma_bl, Pi = black_litterman(W_MKT, Sigma_ann, mu_views_ann, var_views_ann)

# ── Display comparison: Prior vs Views vs Posterior ──────────────────────────
bl_table = pd.DataFrame({
    'Prior Π (CAPM)':   Pi,
    'ARIMA View Q':     mu_views_ann,
    'Posterior μ_BL':   mu_bl,
}, index=ALL_TICKERS).round(4)

print("══ Black-Litterman: Prior vs Views vs Posterior (annualised) ══")
display(bl_table)

In [ ]:
# ── 5.2  Optimal BL portfolio (Max Sharpe on posterior μ_BL, Σ_BL) ────────────
res_bl = minimize(neg_sharpe, np.ones(N_ASSETS) / N_ASSETS,
                  args=(mu_bl, Sigma_bl),
                  method='SLSQP', bounds=BOUNDS, constraints=CONSTRAINTS,
                  options={'maxiter': 500})
w_bl = res_bl.x

print("══ Black-Litterman Optimal Portfolio ══")
print(f"  Annualised Return : {port_return(w_bl, mu_bl)*100:.2f} %")
print(f"  Annualised Vol    : {port_vol(w_bl, Sigma_bl)*100:.2f} %")
print(f"  Sortino Ratio     : see Section 8\n")

fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(N_ASSETS)
width = 0.28
ax.bar(x - width,   W_MKT,   width, label='Market Cap Prior', alpha=0.8, color='steelblue')
ax.bar(x,           w_gmvp,  width, label='GMVP (Markowitz)', alpha=0.8, color='mediumseagreen')
ax.bar(x + width,   w_bl,    width, label='Black-Litterman',  alpha=0.8, color='tomato')
ax.set_xticks(x)
ax.set_xticklabels(ALL_TICKERS, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Weight')
ax.set_title('Portfolio Weights: Market Prior vs GMVP vs Black-Litterman', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. Fama-French Three-Factor Model

Decompone il rendimento del portafoglio in:

| Fattore | Simbolo | Significato |
|---------|---------|-------------|
| Market excess return | **Mkt-RF** | Beta di mercato — rischio sistematico non diversificabile |
| Small-minus-Big | **SMB** | Premio dimensionale (small cap > large cap storicamente) |
| High-minus-Low | **HML** | Premio valore (value stocks > growth stocks storicamente) |
| **Alpha** | **α** | Rendimento non spiegato dai 3 fattori → abilità del gestore |

**Obiettivo:** verificare se i rendimenti del portafoglio BL sono spiegati da rischio sistematico (α≈0) o generano alfa genuino.

In [ ]:
# ── 6.1  Download Fama-French 3-Factor daily data ────────────────────────────
ff3 = fetch_ff3_daily(start='2020-01-01')
if ff3 is not None:
    print(f"FF3 factors: {ff3.shape}  |  {ff3.index[0].date()} → {ff3.index[-1].date()}")
    FF_AVAILABLE = True
else:
    print("FF3 not available — using SPY as market proxy.")
    FF_AVAILABLE = False

# ── 6.2  Build BL portfolio daily returns ────────────────────────────────────
port_ret_d = (log_ret_d * w_bl).sum(axis=1).rename('Port_BL')
port_ret_d.index = pd.to_datetime(port_ret_d.index)

if FF_AVAILABLE:
    df_ff = ff3.join(port_ret_d, how='inner').dropna()
    df_ff['Excess_Port'] = df_ff['Port_BL'] - df_ff['RF']
    X_ff = add_constant(df_ff[['Mkt-RF', 'SMB', 'HML']])
    y_ff = df_ff['Excess_Port']
else:
    # Fallback: single-factor CAPM using SPY as market
    spy_ret  = log_ret_d['SPY'].rename('Mkt-RF')
    rf_daily = (1 + RISK_FREE_ANNUAL) ** (1/252) - 1
    df_ff = pd.DataFrame({'Mkt-RF': spy_ret, 'Port_BL': port_ret_d,
                           'RF': rf_daily}).dropna()
    df_ff['Excess_Port'] = df_ff['Port_BL'] - df_ff['RF']
    df_ff['SMB'] = 0.0
    df_ff['HML'] = 0.0
    X_ff = add_constant(df_ff[['Mkt-RF', 'SMB', 'HML']])
    y_ff = df_ff['Excess_Port']

# ── 6.3  OLS regression ───────────────────────────────────────────────────────
ff_model = OLS(y_ff, X_ff).fit(cov_type='HC3')
print("\n══ Fama-French 3-Factor Regression — BL Portfolio ══")
print(ff_model.summary().tables[1])

In [ ]:
# ── 6.4  Factor decomposition bar chart ──────────────────────────────────────
params  = ff_model.params
conf    = ff_model.conf_int()

# statsmodels names intercept 'const'
factors = [f for f in ['const', 'Mkt-RF', 'SMB', 'HML'] if f in params.index]
labels  = {'const': 'Alpha (α)', 'Mkt-RF': 'Market β', 'SMB': 'SMB (Size)', 'HML': 'HML (Value)'}

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar([labels[f] for f in factors],
       [params[f] for f in factors],
       color=['gold' if f == 'const' else 'steelblue' for f in factors],
       alpha=0.85)
err_low  = [params[f] - conf.loc[f, 0] for f in factors]
err_high = [conf.loc[f, 1] - params[f] for f in factors]
ax.errorbar([labels[f] for f in factors], [params[f] for f in factors],
            yerr=[err_low, err_high], fmt='none', color='black', capsize=5)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Fama-French 3-Factor Decomposition — BL Portfolio', fontweight='bold')
ax.set_ylabel('Coefficient (daily)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()
print(f"R² = {ff_model.rsquared:.4f}  |  Adj-R² = {ff_model.rsquared_adj:.4f}")

---
## 7. Monte Carlo Simulation

Genera **10 000 portafogli random** per visualizzare la nuvola dei portafogli fattibili e confrontarla con la frontiera efficiente e i portafogli ottimali.  
Viene anche eseguito uno **stress test**: shock di volatilità +50% per simulare condizioni di mercato avverse.

In [ ]:
# ── 7.1  Generate 10 000 random portfolios ────────────────────────────────────
np.random.seed(42)
mc_weights  = np.random.dirichlet(np.ones(N_ASSETS), size=N_MC)
mc_returns  = mc_weights @ mu_ann
mc_vols     = np.sqrt(np.einsum('ij,jk,ik->i', mc_weights, Sigma_ann, mc_weights))
mc_sharpe   = (mc_returns - RISK_FREE_ANNUAL) / mc_vols

# ── 7.2  Sortino ratio helper (Monte Carlo) ───────────────────────────────────
def sortino_mc(w, mu, Sigma, rf=RISK_FREE_ANNUAL, n_sim=5_000):
    """Monte Carlo Sortino: simulate portfolio returns and compute downside std."""
    port_mu  = w @ mu
    port_sig = port_vol(w, Sigma)
    sim_ret  = np.random.normal(port_mu, port_sig, n_sim)
    downside = sim_ret[sim_ret < rf]
    if len(downside) < 10:
        return np.nan
    dd_std = np.std(downside)
    return (port_mu - rf) / dd_std if dd_std > 0 else np.nan

mc_sortino = np.array([
    sortino_mc(mc_weights[i], mu_ann, Sigma_ann) for i in range(N_MC)
])

print(f"MC portfolios generated: {N_MC}")
print(f"Return range : {mc_returns.min()*100:.2f}% – {mc_returns.max()*100:.2f}%")
print(f"Vol range    : {mc_vols.min()*100:.2f}% – {mc_vols.max()*100:.2f}%")
print(f"Sharpe range : {mc_sharpe.min():.3f} – {mc_sharpe.max():.3f}")

In [ ]:
# ── 7.3  Plot Monte Carlo cloud + Efficient Frontier ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, metric, cmap_label in zip(
        axes,
        [mc_sharpe, mc_sortino],
        ['Sharpe Ratio', 'Sortino Ratio']):

    sc = ax.scatter(mc_vols * 100, mc_returns * 100, c=metric,
                    cmap='RdYlGn', s=4, alpha=0.5, vmin=np.nanpercentile(metric, 5),
                    vmax=np.nanpercentile(metric, 95))
    plt.colorbar(sc, ax=ax, label=cmap_label)

    # Efficient frontier overlay
    ax.plot(ef_vols * 100, ef_rets * 100, 'b-', linewidth=2, label='Efficient Frontier')

    # Key portfolios
    for label, w, color, marker in [
        ('GMVP',             w_gmvp, 'limegreen',  '*'),
        ('Max Sharpe',       w_msr,  'white',      'D'),
        ('Black-Litterman',  w_bl,   'cyan',       'P'),
    ]:
        ax.scatter(port_vol(w, Sigma_ann)*100, port_return(w, mu_ann)*100,
                   s=200, color=color, marker=marker, edgecolors='black',
                   linewidths=1, zorder=6, label=label)

    ax.set_xlabel('Annualised Volatility (%)')
    ax.set_ylabel('Annualised Return (%)')
    ax.set_title(f'Monte Carlo — {cmap_label}', fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)

plt.suptitle('Monte Carlo Portfolio Cloud (10 000 Simulations)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 7.4  Stress Test: +50% volatility shock ───────────────────────────────────
STRESS_FACTOR = 1.5
Sigma_stress  = Sigma_ann * (STRESS_FACTOR ** 2)  # cov scales with σ²

mc_vols_stress = np.sqrt(np.einsum('ij,jk,ik->i', mc_weights, Sigma_stress, mc_weights))

portfolios = {
    'GMVP':             (port_vol(w_gmvp, Sigma_ann),    port_vol(w_gmvp, Sigma_stress)),
    'Max Sharpe':       (port_vol(w_msr,  Sigma_ann),    port_vol(w_msr,  Sigma_stress)),
    'Black-Litterman':  (port_vol(w_bl,   Sigma_ann),    port_vol(w_bl,   Sigma_stress)),
    'Equal-Weight':     (port_vol(np.ones(N_ASSETS)/N_ASSETS, Sigma_ann),
                         port_vol(np.ones(N_ASSETS)/N_ASSETS, Sigma_stress)),
}

stress_df = pd.DataFrame(portfolios, index=['Normal Vol (%)', 'Stressed Vol (+50%)']) * 100
print("══ Stress Test: Volatility under +50% Shock ══")
display(stress_df.round(2))

---
## 8. Rebalancing Engine — 254-Hour Cycle

L'engine simula il ribilanciamento del portafoglio **ogni 254 ore** su dati storici orari:

1. Calcola pesi BL aggiornati con ARIMA/GARCH sulla finestra precedente
2. Applica i nuovi pesi all'inizio del periodo successivo
3. Accumula i rendimenti e calcola le metriche di performance

In [ ]:
# ── 8.1  Rebalancing engine ───────────────────────────────────────────────────
# Nota: il modello predittivo è calibrato su 254 ore (ARIMA/GARCH horizon).
# La simulazione walk-forward usa finestre di 60 ore per velocità computazionale;
# in produzione si sostituisce SIM_REBALANCE_H = REBALANCE_HOURS (254).

SIM_REBALANCE_H = 60   # ore tra un ribilanciamento e l'altro nella simulazione

def rebalance_bl(log_ret_h, w_mkt, rebalance_h=SIM_REBALANCE_H,
                 lookback=500, tau=TAU, delta=DELTA):
    """
    Walk-forward simulation with Black-Litterman rebalancing every `rebalance_h` hours.
    Returns portfolio hourly log-returns series.
    """
    n = len(log_ret_h)
    port_rets  = []
    weights_log = []

    # Start from the first full lookback window
    current_w = np.ones(N_ASSETS) / N_ASSETS  # equal-weight initially

    for t in range(lookback, n, rebalance_h):
        # ── 1. Estimation window ──────────────────────────────────────────────
        window = log_ret_h.iloc[max(0, t - lookback): t]
        if len(window) < 50:
            break

        # ── 2. Estimate covariance (annualised) ───────────────────────────────
        S = window.cov().values * HOURS_PER_YEAR

        # ── 3. Mean return as view (annualised) ───────────────────────────────
        mu_w  = window.mean().values * HOURS_PER_YEAR
        var_w = (window.std().values ** 2) * HOURS_PER_YEAR

        # ── 4. Black-Litterman posterior ──────────────────────────────────────
        try:
            mu_post, S_post, _ = black_litterman(w_mkt, S, mu_w, var_w, tau, delta)
        except np.linalg.LinAlgError:
            mu_post, S_post = mu_w, S

        # ── 5. Max Sharpe with BL posterior ──────────────────────────────────
        res = minimize(neg_sharpe, current_w,
                       args=(mu_post, S_post), method='SLSQP',
                       bounds=BOUNDS, constraints=CONSTRAINTS,
                       options={'maxiter': 300, 'ftol': 1e-8})
        current_w = res.x if res.success else current_w

        # ── 6. Apply weights to next rebalance period ─────────────────────────
        period = log_ret_h.iloc[t: t + rebalance_h]
        period_port = (period * current_w).sum(axis=1)
        port_rets.append(period_port)
        weights_log.append((period.index[0], current_w.copy()))

    if not port_rets:
        return pd.Series(dtype=float), []
    return pd.concat(port_rets), weights_log

print(f"Running walk-forward rebalancing (every {SIM_REBALANCE_H}h) …")
port_rets_bl, weights_history = rebalance_bl(log_ret_h, W_MKT)

# Benchmarks (same index slice)
idx_common     = port_rets_bl.index
port_rets_ew   = log_ret_h.mean(axis=1).reindex(idx_common).dropna()
port_rets_gmvp = (log_ret_h * w_gmvp).sum(axis=1).reindex(idx_common).dropna()
port_rets_bl   = port_rets_bl.reindex(idx_common).dropna()

print(f"Done: {len(port_rets_bl)} hourly obs · {len(weights_history)} rebalancing events")

In [ ]:
# ── 8.2  Sortino Ratio & performance metrics ──────────────────────────────────

def sortino_ratio(rets, rf_hourly=RF_HOURLY):
    excess    = rets - rf_hourly
    downside  = rets[rets < rf_hourly]
    dd_std    = downside.std()
    return (excess.mean() / dd_std) * np.sqrt(HOURS_PER_YEAR) if dd_std > 0 else np.nan

def sharpe_ratio(rets, rf_hourly=RF_HOURLY):
    excess = rets - rf_hourly
    return (excess.mean() / rets.std()) * np.sqrt(HOURS_PER_YEAR)

def max_drawdown(rets):
    cum = (1 + rets).cumprod()
    peak = cum.cummax()
    dd   = (cum - peak) / peak
    return dd.min()

def calmar_ratio(rets, rf_hourly=RF_HOURLY):
    ann_ret = rets.mean() * HOURS_PER_YEAR
    mdd     = abs(max_drawdown(rets))
    return (ann_ret - RISK_FREE_ANNUAL) / mdd if mdd > 0 else np.nan

perf = {}
for label, r in [('BL (60h rebal)', port_rets_bl),
                 ('GMVP',           port_rets_gmvp),
                 ('Equal-Weight',   port_rets_ew)]:
    perf[label] = {
        'Ann. Return (%)':  round(r.mean() * HOURS_PER_YEAR * 100, 2),
        'Ann. Vol (%)':     round(r.std()  * np.sqrt(HOURS_PER_YEAR) * 100, 2),
        'Sharpe':           round(sharpe_ratio(r), 4),
        'Sortino':          round(sortino_ratio(r), 4),
        'Max Drawdown (%)': round(max_drawdown(r) * 100, 2),
        'Calmar':           round(calmar_ratio(r), 4),
    }

perf_df = pd.DataFrame(perf).T
print("══ Performance Dashboard ══")
display(perf_df)

In [ ]:
# ── 8.3  Cumulative return chart ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# Cumulative returns
ax1 = axes[0]
for label, r, color in [
    ('BL (60h rebal)', port_rets_bl,   'tomato'),
    ('GMVP',           port_rets_gmvp, 'mediumseagreen'),
    ('Equal-Weight',   port_rets_ew,   'steelblue'),
]:
    cum = (1 + r).cumprod()
    ax1.plot(cum.index, cum.values, label=label, linewidth=1.5, color=color)

ax1.set_title('Cumulative Portfolio Returns (Walk-Forward Simulation)', fontweight='bold')
ax1.set_ylabel('Portfolio Value (start = 1)')
ax1.legend()
ax1.grid(alpha=0.3)

# Rolling Sortino (168h = 1 week)
ax2 = axes[1]
for label, r, color in [
    ('BL (60h rebal)', port_rets_bl,   'tomato'),
    ('GMVP',           port_rets_gmvp, 'mediumseagreen'),
]:
    roll_sortino = r.rolling(168).apply(
        lambda x: sortino_ratio(pd.Series(x)), raw=False)
    ax2.plot(roll_sortino.index, roll_sortino.values,
             label=f'{label} — Rolling Sortino (7d)', linewidth=1.2, color=color)

ax2.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax2.set_title('Rolling 7-Day Sortino Ratio', fontweight='bold')
ax2.set_ylabel('Sortino Ratio')
ax2.legend()
ax2.grid(alpha=0.3)
ax2.set_ylim(-20, 20)

plt.tight_layout()
plt.show()

In [ ]:
# ── 8.4  Weight evolution over rebalancing cycles ────────────────────────────
if weights_history:
    dates_w  = [d for d, _ in weights_history]
    w_matrix = np.array([w for _, w in weights_history])

    fig, ax = plt.subplots(figsize=(14, 5))
    bottom = np.zeros(len(dates_w))
    for i, t in enumerate(ALL_TICKERS):
        ax.bar(range(len(dates_w)), w_matrix[:, i], bottom=bottom, label=t, alpha=0.85)
        bottom += w_matrix[:, i]

    step = max(1, len(dates_w) // 10)
    ax.set_xticks(range(0, len(dates_w), step))
    ax.set_xticklabels([str(dates_w[j])[:13] for j in range(0, len(dates_w), step)],
                        rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Portfolio Weight')
    ax.set_title('Black-Litterman Portfolio Weights — Evolution Over Rebalancing Cycles',
                 fontweight='bold')
    ax.legend(fontsize=7, ncol=4, loc='upper center', bbox_to_anchor=(0.5, -0.25))
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()

---
## 9. Conclusioni

### Risultati chiave

| Dimensione | Risultato |
|-----------|-----------|
| **GMVP** | Portafoglio a varianza minima — massima conservazione del capitale, sacrifica rendimento |
| **Black-Litterman** | Pesi stabilizzati rispetto a Markowitz puro: non ci sono allocazioni estreme; le views ARIMA spostano il prior CAPM verso le asset class con momentum positivo |
| **Fama-French** | L'alpha del portafoglio BL — se positivo e statisticamente significativo — indica valore aggiunto oltre il rischio di mercato, size e value; con un universo misto crypto/equity l'R² sarà relativamente basso (crypto non è spiegata da FF) |
| **Monte Carlo** | La nuvola mostra che quasi tutti i portafogli random sono sotto la frontiera efficiente; BL e GMVP si trovano sulla curva o vicino ad essa |
| **Sortino vs Sharpe** | Il Sortino ratio è più alto dello Sharpe per portafogli con rendimenti asimmetrici positivi (es. crypto in bull market): penalizza solo la volatilità al ribasso |

### Limitazioni
- **Crypto 24/7**: la matrice di covarianza mista equity/crypto sovrastima la correlazione durante le ore di chiusura dei mercati → considerare una covariance matrix separata o *DCC-GARCH*
- **ARIMA su rendimenti finanziari**: i rendimenti sono quasi martingale — le previsioni di media sono deboli. Il valore aggiunto del modello è principalmente nella stima della varianza (GARCH)
- **Dati storici 730 giorni**: il lookback limit di yfinance per i dati orari limita la stima della covarianza; Bloomberg consentirebbe serie storiche multi-anno
- **Costi di transazione**: il rebalancing frequente (60h) genera turnover elevato — in produzione aggiungere un vincolo di turnover massimo del 10-20% per ciclo

### Estensioni consigliate
1. **LSTM** per la previsione dei rendimenti a 254 ore (cattura non-linearità)
2. **DCC-GARCH** per covarianze dinamiche
3. **CVaR optimization** al posto di varianza (più robusto a code spesse)
4. **Regime switching** (HMM) per adattare i pesi in bull/bear market